In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
LABEL_MAP = {"nv": 0, "mel": 1}
CLASS_NAMES = ["nv", "mel"]


def build_transforms(train: bool, strong_aug: bool = False) -> transforms.Compose:
    if train and strong_aug:
        return transforms.Compose([
            transforms.RandomRotation(180),
            transforms.RandomResizedCrop(224, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    if train:
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class MelanomaDataset(Dataset):
    def __init__(self, manifest_df: pd.DataFrame, base_dir: Path, transform=None):
        self.df = manifest_df.reset_index(drop=True)
        self.base_dir = Path(base_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = self.base_dir / row["path"].replace("\\", "/")
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = LABEL_MAP[row["dx"]]
        return img, label


def load_split_manifest(base_dir: Path, split_index_path: str | None = None) -> pd.DataFrame:
    base_dir = Path(base_dir)
    path = Path(split_index_path) if split_index_path else base_dir / "data" / "processed" / "split_index.csv"
    return pd.read_csv(path)


def build_datasets(base_dir: Path, split_index_path: str | None = None,
                    limit_per_split: int | None = None,
                    use_precomputed_aug: bool = True,
                    strong_aug: bool = False) -> dict[str, MelanomaDataset]:
    manifest = load_split_manifest(base_dir, split_index_path)

    datasets = {}
    for split in ("train", "val", "test"):
        split_df = manifest[manifest["split"] == split].copy()
        if split != "train" or not use_precomputed_aug:
            split_df = split_df[~split_df["is_augmented"]]
        if limit_per_split:
            per_class = max(1, limit_per_split // 2)
            parts = [group.head(per_class) for _, group in split_df.groupby("dx")]
            split_df = pd.concat(parts, ignore_index=True)
        transform = build_transforms(train=(split == "train"), strong_aug=strong_aug)
        datasets[split] = MelanomaDataset(split_df, base_dir, transform=transform)
    return datasets


def build_dataloaders(datasets: dict[str, MelanomaDataset], batch_size: int = 32,
                       num_workers: int = 2) -> dict[str, DataLoader]:
    loaders = {}
    for split, ds in datasets.items():
        loaders[split] = DataLoader(
            ds, batch_size=batch_size, shuffle=(split == "train"),
            num_workers=num_workers, pin_memory=torch.cuda.is_available(),
        )
    return loaders

In [ ]:
IS_KAGGLE = os.path.exists("/kaggle/input")

SMOKE_TEST = not IS_KAGGLE
ARCHITECTURE = "resnet50"
SEED = 42

EPOCHS_STAGE1 = 1 if SMOKE_TEST else 5
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 12
PATIENCE = 3
BATCH_SIZE = 4 if SMOKE_TEST else 32
LIMIT_PER_SPLIT = 40 if SMOKE_TEST else None
NUM_WORKERS = 0 if SMOKE_TEST else 4

MEL_EXTRA_MULTIPLIER = 1.0
TARGET_RECALL_MEL = 0.90

print(f"IS_KAGGLE={IS_KAGGLE} | SMOKE_TEST={SMOKE_TEST} | arquitetura={ARCHITECTURE} | batch_size={BATCH_SIZE}")

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch import optim
from torchvision import models

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
import os
caminho = "/kaggle/input/datasets/sofyaoliveiras/tonare-tcc1/processed"
print(os.path.exists(caminho))
if os.path.exists(caminho):
    print(os.listdir(caminho))

In [ ]:
if IS_KAGGLE:
    KAGGLE_PROCESSED = "/kaggle/input/datasets/sofyaoliveiras/tonare-tcc1/processed"
    os.makedirs("/kaggle/working/data", exist_ok=True)
    link = "/kaggle/working/data/processed"
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(KAGGLE_PROCESSED, link)
    BASE_DIR = Path("/kaggle/working")
    OUT_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path("..")
    OUT_DIR = BASE_DIR / "data" / "processed"

SPLIT_INDEX_PATH = BASE_DIR / "data" / "processed" / "split_index_combined.csv"
assert SPLIT_INDEX_PATH.exists(), (
    f"{SPLIT_INDEX_PATH} não encontrado - confirme que a nova versão do dataset "
    "(com split_index_combined.csv) está anexada."
)
print("BASE_DIR:", BASE_DIR, "| OUT_DIR:", OUT_DIR, "| manifesto:", SPLIT_INDEX_PATH.name)

In [ ]:
datasets = build_datasets(base_dir=BASE_DIR, split_index_path=SPLIT_INDEX_PATH,
                          limit_per_split=LIMIT_PER_SPLIT, use_precomputed_aug=False, strong_aug=True)
loaders = build_dataloaders(datasets, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

for split, ds in datasets.items():
    contagem = ds.df["dx"].value_counts().to_dict()
    print(f"{split}: {len(ds)} imagens {contagem}")

In [ ]:
imgs, labels = next(iter(loaders["train"]))
print("Shape do batch de imagens:", imgs.shape)
print("Labels do batch:", labels.tolist())
print("Range de valores:", float(imgs.min()), "a", float(imgs.max()))

assert imgs.shape[1:] == (3, 224, 224), "dataloader não está entregando imagens 224x224x3"
print("\nDataloader validado: uma iteração de teste passou sem erro.")

In [ ]:
def build_model(architecture: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    if architecture == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif architecture == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Arquitetura não suportada: {architecture}")
    return model


def set_backbone_trainable(model: nn.Module, architecture: str, trainable: bool) -> None:
    for param in model.parameters():
        param.requires_grad = trainable
    head = model.fc if architecture == "resnet50" else model.classifier
    for param in head.parameters():
        param.requires_grad = True


model = build_model(ARCHITECTURE).to(DEVICE)
set_backbone_trainable(model, ARCHITECTURE, trainable=False)

n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis (Estágio 1): {n_treinaveis:,} de {n_total:,} total")

In [ ]:
train_labels = datasets["train"].df["dx"].map(LABEL_MAP).values
base_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=train_labels)
class_weights = base_weights.copy()
class_weights[1] *= MEL_EXTRA_MULTIPLIER
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))

print(f"Distribuição do treino: {pd.Series(train_labels).value_counts().to_dict()} (0=nv, 1=mel)")
print(f"Pesos da loss: nv={class_weights[0]:.3f}, mel={class_weights[1]:.3f}")
if not SMOKE_TEST:
    assert class_weights[1] > class_weights[0], "peso de mel deveria ser maior que o de nv"

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None) -> dict:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(F.softmax(outputs.detach(), dim=1)[:, 1].cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    labels_arr, probs_arr = np.array(all_labels), np.array(all_probs)
    preds = (probs_arr >= 0.5).astype(int)
    report = classification_report(labels_arr, preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    auc = roc_auc_score(labels_arr, probs_arr) if len(set(all_labels)) > 1 else float("nan")
    return {"loss": total_loss / len(loader.dataset), "accuracy": report["accuracy"],
            "recall_mel": report["mel"]["recall"], "auc": auc,
            "labels": labels_arr, "probs": probs_arr}


def train_stage(model, loaders, criterion, optimizer, n_epochs: int, stage_name: str, patience: int = PATIENCE):
    history, best_auc, best_state, sem_melhora = [], -1.0, None, 0
    print(f"[{stage_name}] pesos da loss (nv, mel): {[round(w, 3) for w in criterion.weight.tolist()]}")
    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(model, loaders["train"], criterion, optimizer)
        va = run_epoch(model, loaders["val"], criterion, optimizer=None)
        print(f"[{stage_name}] época {epoch}/{n_epochs} | "
              f"treino: loss={tr['loss']:.3f} auc={tr['auc']:.3f} | "
              f"val: loss={va['loss']:.3f} auc={va['auc']:.3f} recall_mel@0.5={va['recall_mel']:.3f}")
        history.append({"epoch": epoch, "train": tr, "val": va})

        if va["auc"] > best_auc:
            best_auc, best_state, sem_melhora = va["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            sem_melhora += 1
            if sem_melhora >= patience:
                print(f"[{stage_name}] early stopping: sem melhora da AUC de validação há {patience} épocas")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[{stage_name}] modelo restaurado para a melhor época (val AUC = {best_auc:.3f})")
    return history

In [ ]:
optimizer_stage1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)
history_stage1 = train_stage(model, loaders, criterion, optimizer_stage1, EPOCHS_STAGE1, "Estágio 1")

In [ ]:
set_backbone_trainable(model, ARCHITECTURE, trainable=True)
n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis (Estágio 2): {n_treinaveis:,}")

optimizer_stage2 = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
history_stage2 = train_stage(model, loaders, criterion, optimizer_stage2, EPOCHS_STAGE2, "Estágio 2")

In [ ]:
def sens_spec_prec(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    return sens, spec, prec


val_res = run_epoch(model, loaders["val"], criterion)
val_labels, val_probs = val_res["labels"], val_res["probs"]
print(f"Validação: AUC-ROC = {val_res['auc']:.3f}\n")

print(f"{'limiar':>7} {'sensibilidade(mel)':>19} {'especificidade':>15} {'precisão(mel)':>14}")
for t in np.arange(0.05, 0.96, 0.05):
    sens, spec, prec = sens_spec_prec(val_labels, val_probs, t)
    print(f"{t:>7.2f} {sens:>19.3f} {spec:>15.3f} {prec:>14.3f}")

In [ ]:
thresholds = np.round(np.arange(0.01, 0.99, 0.01), 2)
sens_por_t = {t: sens_spec_prec(val_labels, val_probs, t)[0] for t in thresholds}
candidatos = [t for t, s in sens_por_t.items() if s >= TARGET_RECALL_MEL]

if candidatos:
    best_threshold = max(candidatos)
    print(f"Limiar escolhido na validação: {best_threshold:.2f} (sensibilidade >= {TARGET_RECALL_MEL})")
else:
    best_threshold = max(sens_por_t, key=lambda t: (sens_por_t[t], t))
    print(f"AVISO: nenhum limiar atinge sensibilidade {TARGET_RECALL_MEL} na validação. "
          f"Usando o de maior sensibilidade: {best_threshold:.2f} ({sens_por_t[best_threshold]:.3f})")

test_res = run_epoch(model, loaders["test"], criterion)
test_labels, test_probs = test_res["labels"], test_res["probs"]
print(f"\nTESTE — AUC-ROC = {test_res['auc']:.3f}")
for nome, t in [("limiar padrão 0.50", 0.5), (f"limiar escolhido {best_threshold:.2f}", best_threshold)]:
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    tn, fp, fn, tp = confusion_matrix(test_labels, (test_probs >= t).astype(int), labels=[0, 1]).ravel()
    print(f"  {nome}: sensibilidade(mel)={sens:.3f} especificidade={spec:.3f} precisão(mel)={prec:.3f} "
          f"| melanomas perdidos: {fn} de {tp + fn} | nevos marcados como mel: {fp} de {tn + fp}")

In [ ]:
checkpoint_path = Path("/kaggle/working") / f"model_{ARCHITECTURE}_baseline.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": ARCHITECTURE,
    "label_map": LABEL_MAP,
    "threshold": float(best_threshold),
    "target_recall_mel": TARGET_RECALL_MEL,
    "mel_extra_multiplier": MEL_EXTRA_MULTIPLIER,
    "val_auc": float(val_res["auc"]),
    "smoke_test": SMOKE_TEST,
}, checkpoint_path)
print(f"Checkpoint salvo em: {checkpoint_path}")